# 01 · Extracción de Series BCRP (datos crudos)
**Tesis:** Medición del ciclo financiero en Perú mediante técnicas de reducción dimensional y machine learning
**Autor:** Roberto Samaniego
**Asesor:** Dr. Sergio Camiz

---

**Responsabilidad única de este notebook:** descargar las series desde la API del BCRP
y guardarlas exactamente como llegan, sin ninguna transformación analítica
(sin resampleo, sin relleno de nulos, sin logaritmos ni diferencias, sin estandarización).

El único procesamiento que se aplica aquí es el estrictamente necesario para poder
leer el dato (parseo de fecha y de número decimal) — no para modificar su contenido.

**Salida:** `data/raw/series_bcrp_raw.csv` — este archivo se considera *inmutable* a partir
de aquí. Ningún notebook posterior debe sobrescribirlo.

## 1. Librerías

In [1]:
import os
os.chdir('..')  # sube un nivel desde notebooks/ a la raíz del repo
print(f"Directorio: {os.getcwd()}")

Directorio: /Users/robert/Documents/tesis-ciclo-financiero-peru


In [2]:
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas')

Librerías cargadas


## 2. Definición de series y función de descarga

Se mantiene el mismo diccionario de códigos de serie usado en las versiones previas
del proyecto (unificando `01_exploracion_bcrp.ipynb` y `02_preprocesamiento.ipynb`).

In [3]:
# Códigos de series del BCRP (mensuales, salvo el índice de inmuebles que es trimestral
# y se carga aparte en la sección 4)
#
# CORRECCIÓN (verificada contra BCRPData): los códigos PN01208PM/PN01209PM/PN01210PM
# NO son series de crédito -- son "Tipo de Cambio Bancario Compra/Venta/Promedio".
# Se reemplazan por el desglose real de crédito por tipo (Cuadro 19 del BCRP:
# "Crédito al sector privado de las sociedades de depósito, por tipo de crédito
# y por moneda"). Además se elimina 'Crédito bancario SP' (PN00522MM) por ser
# un subconjunto casi colineal de 'Crédito SF sector privado' (PN00518MM).
#
# Nota de colinealidad: Total ≈ Empresarial + Consumo + Hipotecario.
# No uses el total junto con los tres componentes en el mismo PCA/HFC.
SERIES = {
    # CRÉDITO (desglose por tipo, en soles, ambas monedas -- Cuadro 19 BCRP)
    'Crédito SF sector privado': 'PN00518MM',  # Total
    'Crédito empresarial':       'PN00532MM',  # Saldos - A Empresas
    'Crédito consumo':           'PN00533MM',  # Saldos - Consumo
    'Crédito hipotecario':       'PN00534MM',  # Saldos - Hipotecario
    # TASAS
    'Tasa activa TAMN':          'PN07807NM',
    'Tasa pasiva TIPMN':         'PN07816NM',
    'Tasa referencia BCRP':      'PD04722MM',
    # TIPO DE CAMBIO Y MERCADO
    'Tipo de cambio':            'PN01246PM',
    'Índice BVL':                'PN01142MM',
    'IPC Lima':                  'PN38705PM',
    # ACTIVIDAD
    'PBI desestacionalizado':    'PN01773AM',
    'Demanda interna':           'PN01774AM',
    'Reservas internacionales':  'PN00027MM',
    # LIQUIDEZ (equivalentes oficiales BCRP a M1/M2/M3 -- nomenclatura real del BCRP)
    'Liquidez M1':               'PN00199MM',  # Dinero (circulante + depositos vista)
    'Liquidez M2':               'PN00208MM',  # Liquidez en Soles (M1 + cuasidinero MN)
    'Liquidez M3':               'PN00214MM',  # Liquidez Total (soles + dolares)
    # TASA ADICIONAL
    'Tasa interbancaria':        'PN07819NM',  # Tasa Interbancaria Promedio
    # INFLACION SUBYACENTE
    'IPC Subyacente':            'PN38708PM',  # IPC Subyacente (Indice Dic.2021=100)
}

## 3. Descarga de todas las series (sin resamplear, sin imputar)

In [4]:
import requests
import pandas as pd

BASE_URL = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api'

def descargar_serie(codigo, inicio='2007-1', fin='2025-12'):
    """
    Descarga una serie del BCRP por código y la retorna como Series de pandas
    indexada por fecha. El parseo de fecha/decimal es conversión de formato,
    no altera el valor reportado por el BCRP.
    """
    url = f'{BASE_URL}/{codigo}/json/{inicio}/{fin}/ing'
    r = requests.get(url, timeout=15)
    r.raise_for_status()  # lanza error explícito si el BCRP responde con código != 200
    data = r.json()
    periodos = data.get('periods', [])
    if not periodos:
        raise ValueError(f'La API del BCRP devolvió 0 periodos para {codigo}')
    registros = []
    for p in periodos:
        valor = p['values'][0]
        try:
            valor = float(str(valor).replace(',', '.'))
        except (ValueError, TypeError):
            valor = None
        registros.append({'fecha': p['name'], 'valor': valor})
    df = pd.DataFrame(registros)
    df['fecha'] = pd.to_datetime(df['fecha'], format='mixed', errors='coerce')
    return df.set_index('fecha')['valor']

# --- Descarga de todas las series (sin resamplear, sin imputar) ---
series_descargadas = {}
errores = []

for nombre, codigo in SERIES.items():
    try:
        series_descargadas[nombre] = descargar_serie(codigo)
        print(f'  OK  {nombre} ({codigo}): {len(series_descargadas[nombre])} obs.')
    except Exception as e:
        errores.append((nombre, codigo, str(e)))
        print(f'  ERROR  {nombre} ({codigo}): {e}')

if errores:
    print(f'\nSeries con error: {len(errores)} — revisar antes de continuar')

  OK  Crédito SF sector privado (PN00518MM): 228 obs.
  OK  Crédito empresarial (PN00532MM): 228 obs.
  OK  Crédito consumo (PN00533MM): 228 obs.
  OK  Crédito hipotecario (PN00534MM): 228 obs.
  OK  Tasa activa TAMN (PN07807NM): 228 obs.
  OK  Tasa pasiva TIPMN (PN07816NM): 228 obs.
  OK  Tasa referencia BCRP (PD04722MM): 228 obs.
  OK  Tipo de cambio (PN01246PM): 228 obs.
  OK  Índice BVL (PN01142MM): 228 obs.
  OK  IPC Lima (PN38705PM): 228 obs.
  OK  PBI desestacionalizado (PN01773AM): 228 obs.
  OK  Demanda interna (PN01774AM): 228 obs.
  OK  Reservas internacionales (PN00027MM): 228 obs.
  OK  Liquidez M1 (PN00199MM): 228 obs.
  OK  Liquidez M2 (PN00208MM): 228 obs.
  OK  Liquidez M3 (PN00214MM): 228 obs.
  OK  Tasa interbancaria (PN07819NM): 228 obs.
  OK  IPC Subyacente (PN38708PM): 228 obs.


## 4. Consolidación en tabla ancha

Se unifican todas las series con un `outer join` sobre las fechas: esto preserva
**todas** las fechas de **todas** las series, dejando `NaN` donde una serie no
tiene dato — no se recorta ni se imputa nada en esta etapa.

In [5]:
df_raw = pd.DataFrame(series_descargadas)
df_raw = df_raw.sort_index()

print(f'Dataset crudo consolidado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
print(f'Periodo: {df_raw.index.min().strftime("%Y-%m")} a {df_raw.index.max().strftime("%Y-%m")}')
df_raw.tail()

Dataset crudo consolidado: 228 filas x 18 columnas
Periodo: 2007-01 a 2025-12


,Crédito SF sector privado,Crédito empresarial,Crédito consumo,Crédito hipotecario,Tasa activa TAMN,Tasa pasiva TIPMN,Tasa referencia BCRP,Tipo de cambio,Índice BVL,IPC Lima,PBI desestacionalizado,Demanda interna,Reservas internacionales,Liquidez M1,Liquidez M2,Liquidez M3,Tasa interbancaria,IPC Subyacente
fecha,,,,,,,,,,,,,,,,,,
2025-08-01,470356.301562,247323.020900,104750.3996,72784.2576,15.1490,2.1348,4.50,3.543200,34937.73,115.586828,191.100818,216.143896,87752.818921,158984.423067,516153.852177,702940.601847,4.5079,115.998679
2025-09-01,470208.838774,246752.668765,105563.9020,73326.4468,15.3507,2.1383,4.25,3.502818,38054.76,115.599718,191.867835,210.518623,85148.225305,161111.064693,520987.904396,709175.882797,4.3685,116.016556
2025-10-01,470552.782777,247446.294924,106685.7408,73902.4976,15.5555,2.1200,4.25,3.414091,38599.28,115.484549,192.028747,217.057443,89960.765240,163534.940316,525748.900702,712922.060965,4.2500,116.070864
2025-11-01,474468.883319,249857.815330,107487.5164,74421.5588,15.7477,2.0993,4.25,3.372850,38978.53,115.612485,190.919048,215.534038,90898.183584,170252.343551,534918.802279,721826.606486,4.2478,116.220266
2025-12-01,479089.326795,252432.680596,107850.7452,74632.2836,15.8871,1.9771,4.25,3.366737,43464.77,115.894705,193.429032,242.172683,90214.484300,178926.594941,550978.611986,737697.228516,4.2411,116.492958


## 5. Verificación mínima de la descarga

Solo impresión en pantalla — no se modifica ni se guarda nada distinto de lo descargado.

In [6]:
cobertura = pd.DataFrame({
    'Inicio': df_raw.apply(lambda x: x.dropna().index.min()),
    'Fin':    df_raw.apply(lambda x: x.dropna().index.max()),
    'Obs':    df_raw.count(),
})
print('Cobertura temporal por serie (diagnóstico, no altera datos):')
cobertura

Cobertura temporal por serie (diagnóstico, no altera datos):


,Inicio,Fin,Obs
Crédito SF sector privado,2007-01-01,2025-12-01,228
Crédito empresarial,2007-01-01,2025-12-01,228
Crédito consumo,2007-01-01,2025-12-01,228
Crédito hipotecario,2007-01-01,2025-12-01,228
Tasa activa TAMN,2007-01-01,2025-12-01,228
Tasa pasiva TIPMN,2007-01-01,2025-12-01,228
Tasa referencia BCRP,2007-01-01,2025-12-01,228
Tipo de cambio,2007-01-01,2025-12-01,228
Índice BVL,2007-01-01,2025-12-01,228
IPC Lima,2007-01-01,2025-12-01,228


## 6. Guardar datos crudos (inmutables)

A partir de este punto, `data/raw/series_bcrp_raw.csv` es la fuente de verdad
para todos los notebooks siguientes. Ningún otro notebook debe volver a descargar
ni sobrescribir este archivo salvo que se quiera actualizar la ventana temporal
de la tesis (en cuyo caso se recomienda versionar el archivo, p. ej. con fecha).

In [7]:
os.makedirs('data/raw', exist_ok=True)
df_raw.to_csv('data/raw/series_bcrp_raw.csv')
print('Guardado en data/raw/series_bcrp_raw.csv')
print(f'Dimensiones: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')

Guardado en data/raw/series_bcrp_raw.csv
Dimensiones: 228 filas x 18 columnas


In [8]:
print(series_descargadas)
print(errores)

{'Crédito SF sector privado': fecha
2007-01-01     69819.470429
2007-02-01     70717.513741
2007-03-01     72596.321022
2007-04-01     74195.323389
2007-05-01     75878.925497
                  ...      
2025-08-01    470356.301562
2025-09-01    470208.838774
2025-10-01    470552.782777
2025-11-01    474468.883319
2025-12-01    479089.326795
Name: valor, Length: 228, dtype: float64, 'Crédito empresarial': fecha
2007-01-01     39105.687117
2007-02-01     39559.563157
2007-03-01     40630.071977
2007-04-01     41664.440265
2007-05-01     42288.944366
                  ...      
2025-08-01    247323.020900
2025-09-01    246752.668765
2025-10-01    247446.294924
2025-11-01    249857.815330
2025-12-01    252432.680596
Name: valor, Length: 228, dtype: float64, 'Crédito consumo': fecha
2007-01-01     11831.023259
2007-02-01     12056.909321
2007-03-01     12336.710861
2007-04-01     12608.628826
2007-05-01     13065.238278
                  ...      
2025-08-01    104750.399600
2025-09-01    

In [9]:
df_raw[['Crédito hipotecario', 'Crédito consumo', 'Tipo de cambio']].describe()

,Crédito hipotecario,Crédito consumo,Tipo de cambio
count,228.000000,228.000000,228.000000
mean,39531.196624,56730.937798,3.239051
std,19645.028750,28923.537404,0.398669
min,7881.540785,11831.023259,2.551909
25%,21609.014565,31205.508646,2.847696
50%,40215.286340,55629.202720,3.248648
75%,55499.087900,77820.681600,3.571850
max,74632.283600,107850.745200,4.107477


## 7. Descarga del EMBI Perú (serie diaria, archivo separado)

El EMBI es la única serie **diaria** de este notebook; todas las demás en
`SERIES` son mensuales. Para no romper la asunción de frecuencia mensual del
`df_raw` consolidado en la sección 4, se descarga y guarda por separado,
siguiendo el mismo patrón que `indice_precios_inmuebles_bcrp.csv`: un archivo
raw por frecuencia distinta, combinado más adelante en `04_preprocesamiento.ipynb`
al momento de sincronizar todo a trimestral.

In [10]:
def descargar_serie_diaria(codigo, inicio, fin):
    """
    Igual que descargar_serie(), pero para series diarias la API del BCRP
    necesita el periodo con dia completo (AAAA-M-D), no solo AAAA-M.
    """
    url = f'{BASE_URL}/{codigo}/json/{inicio}/{fin}/ing'
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    data = r.json()
    periodos = data.get('periods', [])
    if not periodos:
        raise ValueError(f'La API del BCRP devolvio 0 periodos para {codigo}')
    registros = []
    for p in periodos:
        valor = p['values'][0]
        try:
            valor = float(str(valor).replace(',', '.'))
        except (ValueError, TypeError):
            valor = None
        registros.append({'fecha': p['name'], 'valor': valor})
    df = pd.DataFrame(registros)
    df['fecha'] = pd.to_datetime(df['fecha'], format='mixed', errors='coerce')
    return df.set_index('fecha')['valor']

print('Descargando EMBI Perú (serie diaria, archivo separado)...')
embi = descargar_serie_diaria('PD04709XD', inicio='2003-1-1', fin='2025-12-31')
df_embi = embi.to_frame(name='embi_peru')

os.makedirs('data/raw', exist_ok=True)
df_embi.to_csv('data/raw/embi_peru_raw.csv')
print(f'EMBI Perú guardado: {df_embi.shape[0]} observaciones diarias')
print('Archivo: data/raw/embi_peru_raw.csv (diario, inmutable, igual que series_bcrp_raw.csv)')

Descargando EMBI Perú (serie diaria, archivo separado)...
EMBI Perú guardado: 6001 observaciones diarias
Archivo: data/raw/embi_peru_raw.csv (diario, inmutable, igual que series_bcrp_raw.csv)
